
# Semantische Suche in arXiv-Artikeln mit LangChain

In dieser Übung verwenden wir arXiv, um echte wissenschaftliche Abstracts abzurufen, sie zu chunken und in einer Vektordatenbank zu speichern; und schließlich darauf semantische Suchen durchzuführen.

Dabei lernen Sie: 
- arXiv-Abstracts abzurufen und zu verarbeiten  
- Texte in Chunks aufzteilen  
- Embeddings zu erzeugen   
- eine Vektordatenbank aufbauen (Chroma)  
- semantische Ähnlichkeitssuche durchzuführen  


## arXiv
arXiv (ausgesprochen **archive**) ist eine Plattform für wissenschaftliche Preprints.

Forschende aus Bereichen wie Physik, Informatik, Mathematik, KI oder Statistik veröffentlichen dort ihre Arbeiten, bevor sie in Fachzeitschriften erscheinen.

- offen, kostenlos und ohne Login nutzbar.
- strukturierte Metadaten (Titel, Abstract, Autor\*innen, Veröffentlichungsdatum)
- API-Zugang, dadurch nutzbar für Text- und Datenanalysen


## LangChain
LangChain ist ein Python-Framework zur Entwicklung von Anwendungen, die mit großen Sprachmodellen (LLMs) oder semantischer Textverarbeitung arbeiten.
Es stellt Bausteine bereit für:

- Datenverarbeitung: Laden, Aufteilen und Strukturieren von Texten
- Embeddings: Umwandlung von Text in numerische Vektoren zur semantischen Analyse
- Vektordatenbanken: Speicherung und Wiederfinden semantisch ähnlicher Inhalte
- (optional) LLMs: Anbindung von Modellen wie GPT, Mistral oder Claude für Textgenerierung und Dialoge

In dieser Übung konzentrieren wir uns ausschließlich auf den Retrieval-Teil – also das Chunken, Erstellen von Embeddings und semantische Suchen, nicht auf die Textgenerierung.

## 0. Installation benötigter Bibliotheken

- Aktivieren Sie Ihre `venv`
- Aktivieren Sie die benötigten Bibliotheken: `pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface chromadb sentence-transformers torch arxiv tqdm`
- Aktualisieren Sie Ihre `requirements.txt`
- Nach der Installation müssen Sie den Kernel des Notebooks ggf. neu starten, damit die neuen Bibliotheken verfügbar sind


## 1. Data Collection mit der arXiv API

1. Überlegen Sie sich einen Suchbegriff und definieren Sie eine entsprechende Variable. 
2. Überlegen Sie, wieviele Treffer Sie erhalten möchten und definieren Sie eine entsprechende Variable. 
3. Übergeben Sie die beiden Variablen als Parameter `query` und `max_results` an das untenstehende Search()-Objekt.

#### Lösung

In [74]:
import arxiv
from tqdm import tqdm

# 1. Suchbegriff definieren
query = 'multimodal' 

# 2. Trefferanzahl definieren
max_results = 10

# 3. Suche ausführen
search = arxiv.Search(query, max_results, sort_by=arxiv.SortCriterion.Relevance)

4. Führen Sie die Suche aus und lassen Sie sich die Trefferliste mit dem vorhandenen Code ausgeben. 
5. Bonus: Ändern Sie die Sortierung der Ergebnisse von `Relevance` auf `SubmittedDate` und führen Sie die Suche erneut aus.

#### Lösung

In [75]:
# Optional: Wir definieren uns eine Funktion, um den Code unten wiederverwenden zu können
def print_hits(search, return_docs=False):
    docs = []
    for result in search.results():
        docs.append({
            "title": result.title,
            "summary": result.summary,
            "url": result.entry_id,
            "published": result.published
        })

    print(f"{len(docs)} Artikel geladen.")
    for d in docs[:2]:
        print(f"\n📄 {d['title']}\n{d['summary'][:250]}...")
    
    # optional, falls wir nicht nur printen, sondern damit weiterarbeiten wollen
    if return_docs: 
        return docs 

In [76]:
# 4. Ausgabe Trefferliste
search = arxiv.Search(query=query, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance)
print_hits(search)

/var/folders/j9/lnrs75px2vvdhgh2j143swp40000gn/T/ipykernel_6247/1910325622.py:4: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


10 Artikel geladen.

📄 Introducing Representations of Facial Affect in Automated Multimodal Deception Detection
Automated deception detection systems can enhance health, justice, and security in society by helping humans detect deceivers in high-stakes situations across medical and legal domains, among others. This paper presents a novel analysis of the discri...

📄 The Multimodal Universe: Enabling Large-Scale Machine Learning with 100TB of Astronomical Scientific Data
We present the MULTIMODAL UNIVERSE, a large-scale multimodal dataset of scientific astronomical data, compiled specifically to facilitate machine learning research. Overall, the MULTIMODAL UNIVERSE contains hundreds of millions of astronomical observ...


In [77]:
# 5. Sortierung nach SubmittedDate bei der Ausführung der Suche
search_sorted_by_submit = arxiv.Search(query=query, max_results=max_results, sort_by=arxiv.SortCriterion.SubmittedDate)
docs = print_hits(search_sorted_by_submit, return_docs=True)

/var/folders/j9/lnrs75px2vvdhgh2j143swp40000gn/T/ipykernel_6247/1910325622.py:4: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


10 Artikel geladen.

📄 Video-R2: Reinforcing Consistent and Grounded Reasoning in Multimodal Language Models
Reasoning over dynamic visual content remains a central challenge for multimodal large language models. Recent thinking models generate explicit reasoning traces for interpretability; however, their reasoning often appears convincing while being logi...

📄 Video-CoM: Interactive Video Reasoning via Chain of Manipulations
Recent multimodal large language models (MLLMs) have advanced video understanding, yet most still "think about videos" ie once a video is encoded, reasoning unfolds entirely in text, treating visual input as a static context. This passive paradigm cr...


## 2. Chunking

1. Betrachten Sie den folgenden Code: Welche Chunking-Methode wird hier eingesetzt? 
2. Passen Sie den Code so an, dass Sliding Window berücksichtigt wird. 
3. Experimentieren Sie mit verschiedenen Werten für die Größe der Chunks und des Sliding Window. 

In [78]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document

# In LangChain-Dokumente umwandeln
documents = [
    Document(page_content=d["summary"], metadata={"title": d["title"], "url": d["url"]})
    for d in docs
]

splitter = CharacterTextSplitter(
    separator = ' ',
    chunk_size=100, 
    chunk_overlap=0
    )
chunks = splitter.split_documents(documents)

print(f"{len(chunks)} Chunks erzeugt.")

for i, c in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(c.page_content)

153 Chunks erzeugt.

--- Chunk 1 ---
Reasoning over dynamic visual content remains a central challenge for multimodal large language

--- Chunk 2 ---
models. Recent thinking models generate explicit reasoning traces for interpretability; however,

--- Chunk 3 ---
their reasoning often appears convincing while being logically inconsistent or weakly grounded in


### Lösung:

In [79]:
# Andere chunk_size wählen
chunk_size = 150

# chunk_overlap einen Wert > 0, z.B. 10 zuweisen, um Sliding Window zu setzen
chunk_overlap = 10

In [80]:
documents = [
    Document(page_content=d["summary"], metadata={"title": d["title"], "url": d["url"]})
    # Wir nutzen hier die Variable docs, die oben von print_hits() zurückgegeben wurde
    for d in docs
]

splitter = CharacterTextSplitter(
    separator = ' ',
    chunk_size=chunk_size, 
    chunk_overlap=chunk_overlap
    )
chunks = splitter.split_documents(documents)

print(f"{len(chunks)} Chunks erzeugt.")

for i, c in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i+1} ---")
    print(c.page_content)

109 Chunks erzeugt.

--- Chunk 1 ---
Reasoning over dynamic visual content remains a central challenge for multimodal large language models. Recent thinking models generate explicit

--- Chunk 2 ---
explicit reasoning traces for interpretability; however, their reasoning often appears convincing while being logically inconsistent or weakly

--- Chunk 3 ---
or weakly grounded in visual evidence. We identify and formalize these issues through two diagnostic metrics: Think Answer Consistency (TAC), which

--- Chunk 4 ---
which measures the alignment between reasoning and answers, and Video Attention Score (VAS), which captures the extent to which reasoning depends on

--- Chunk 5 ---
depends on visual versus textual cues. Analysis across 11 video reasoning benchmarks shows that current models rely heavily on linguistic priors


## 3. Embeddings erzeugen und Vektordatenbank erstellen
1. Wählen Sie ein Embedding-Modell aus, z.B. eines von denen, die wir uns bereits gemeinsam angeschaut haben. 
2. Nutzen Sie das Modell, um Embeddings für die Chunks zu erzeugen und eine Vektordatenbank zu erstellen.

#### Lösung:

In [85]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = HuggingFaceEmbeddings(model_name=model_name)
db = Chroma.from_documents(chunks, embedding=embedding_model, persist_directory="../../Data/arxiv_db")
print("Vektordatenbank erstellt.")

Vektordatenbank erstellt.


In [86]:
# Datenbank leeren
# db.delete_collection()

## 4. Semantische Suche in der Vektordatenbank
1. Überlegen Sie sich eine Frage oder Phrase, mit der Sie nach ähnlichen Dokumenten in der Vektordatenbank suchen und schauen Sie sich die Ergebnisse an. 
2. Experimentieren Sie mit verschiedenen Queries und Werten für `k`. 
3. Wie könnte man Metadaten (z. B. Jahr, Autor:innen) für die Suche nutzen? 


In [94]:
query = 'evaluate' 
results = db.similarity_search(query, k=3)

for hit in results:
    print(f"\n Treffer: {hit.metadata['title']}")
    print(hit.page_content)
    print("Quelle:", hit.metadata["url"])


 Treffer: Optimizing Multimodal Language Models through Attention-based Interpretability
to validate the method's effectiveness. By calculating Head Impact (HI) scores we quantify an attention head's focus on key objects, indicating its
Quelle: http://arxiv.org/abs/2511.23375v1

 Treffer: Optimizing Multimodal Language Models through Attention-based Interpretability
to identify which components are most effective for training to balance efficiency and performance. We propose an attention-based interpretability
Quelle: http://arxiv.org/abs/2511.23375v1

 Treffer: Transformer-Driven Triple Fusion Framework for Enhanced Multimodal Author Intent Classification in Low-Resource Bangla
84.11% macro-F1 score, establishing a new state-of-the-art with an 8.4 percentage-point improvement over prior Bangla multimodal approaches. Our
Quelle: http://arxiv.org/abs/2511.23287v1


In [99]:
# Similarity score  mit ausgeben + anderen Wert für k setzen
query = 'evaluate multimodal LLMs' 
results = db.similarity_search_with_score(query, k=5)

for hit in results:
    chunk = hit[0]
    score = hit[1]

    print(f"\n Treffer: {chunk.metadata['title']}")
    print(chunk.page_content)
    # print("Quelle:", chunk.metadata["url"])
    cos_sim = 1 - score # the returned score is cosine distance
    print("Cosine similarity with query:", cos_sim)


 Treffer: Optimizing Multimodal Language Models through Attention-based Interpretability
these multimodal language models (MLMs) to downstream tasks, full fine-tuning is computationally expensive. Parameter-Efficient Fine-Tuning (PEFT)
Cosine similarity with query: -0.010393500328063965

 Treffer: Chart2Code-MoLA: Efficient Multi-Modal Code Generation via Adaptive Expert Routing
cross-type generalization, memory efficiency, and modular design. To address these challenges, this paper proposes C2C-MoLA, a multimodal framework
Cosine similarity with query: -0.04017806053161621

 Treffer: Chart2Code-MoLA: Efficient Multi-Modal Code Generation via Adaptive Expert Routing
confirm scalability for real-world multimodal code generation.
Cosine similarity with query: -0.08478713035583496

 Treffer: Optimizing Multimodal Language Models through Attention-based Interpretability
objects. We utilize this information to select optimal model components for PEFT in multimodal models. Our contributions

- Wir sehen, dass die Kosinus-Ähnlickeit leicht negativ ist bzw. etwa bei Null liegt.
- Was sagt uns das? 

## 5. Erweiterungen

### 1. Mehr Daten
- Holen Sie sich eine größere Menge an Dokumenten (z.B. `max_results=100`) und erstellen Sie damit eine Vektordatenbank. 
- Holen Sie sich Artikel zu verschiedenen Suchbegriffen und kombinieren Sie diese in einer Vektordatenbank. 

### 2. Andere Chunking-Methode
- Informieren Sie sich über andere Möglichkeiten des Chunkings mit Hilfe der LangChain-Docs: https://docs.langchain.com/oss/python/integrations/splitters
- Probieren Sie mindestens eine weitere Chunking-Methode aus. 

### 3. Verfeinerung der Suche
- Filtern Sie die Ergebnisse nach Erscheinungsjahr

#### 5.1 Lösung

In [107]:
def create_docs(search):
    docs = []
    for result in search.results():
        docs.append({
            "title": result.title,
            "summary": result.summary,
            "url": result.entry_id,
            "published": result.published
        })

    print(f"{len(docs)} Artikel geladen.")
    return docs


def create_chunks(docs):
    documents = [
        Document(page_content=d["summary"], metadata={"title": d["title"], "url": d["url"]})
        # Wir nutzen hier die Variable docs, die oben von print_hits() zurückgegeben wurde
        for d in docs
    ]

    splitter = CharacterTextSplitter(
        separator = ' ',
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap
        )
    chunks = splitter.split_documents(documents)
    print(f"{len(chunks)} Chunks erzeugt.")
    return chunks


def create_vector_store(embedding_model, db_name):
    db = Chroma.from_documents(chunks, embedding=embedding_model, persist_directory=db_name)
    print("Vektordatenbank erstellt.")
    return db

def print_results(results):
    for hit in results:
        chunk = hit[0]
        score = hit[1]
        print(f"\n Treffer: {chunk.metadata['title']}")
        print(chunk.page_content)
        print("Quelle:", chunk.metadata["url"])
        cos_sim = 1 - score # the returned score is cosine distance
        print("Cosine similarity with query:", cos_sim)

In [ ]:
db.delete_collection() 

query = 'multimodal' 
max_results = 100
search = arxiv.Search(query=query, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance)

docs = create_docs(search)
chunks = create_chunks(docs)

embedding_model = HuggingFaceEmbeddings(model_name= "sentence-transformers/all-MiniLM-L6-v2")
db = create_vector_store(embedding_model, db_name='../../Data/arxiv_db_larger')

/var/folders/j9/lnrs75px2vvdhgh2j143swp40000gn/T/ipykernel_6247/3041715250.py:3: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


100 Artikel geladen.
931 Chunks erzeugt.
Vektordatenbank erstellt.


- Wir sehen, dass die Scores nun höher werden
- Aber passen die retrieved chunks besonders gut zur query?

In [117]:
query = 'evaluate multimodal LLMs' 
results = db.similarity_search_with_score(query, k=5)
print_results(results)


 Treffer: Wiki-LLaVA: Hierarchical Retrieval-Augmented Generation for Multimodal LLMs
Multimodal LLMs are the natural evolution of LLMs, and enlarge their capabilities so as to work beyond the pure textual modality. As research is being
Quelle: http://arxiv.org/abs/2404.15406v2
Cosine similarity with query: 0.5436115264892578

 Treffer: LLMs Meet Multimodal Generation and Editing: A Survey
of LLMs in multimodal generation and exhaustively investigate the critical technical components behind these methods and the multimodal datasets
Quelle: http://arxiv.org/abs/2405.19334v2
Cosine similarity with query: 0.5383468866348267

 Treffer: LLMs Meet Multimodal Generation and Editing: A Survey
With the recent advancement in large language models (LLMs), there is a growing interest in combining LLMs with multimodal learning. Previous surveys
Quelle: http://arxiv.org/abs/2405.19334v2
Cosine similarity with query: 0.3567241430282593

 Treffer: LLMs Meet Multimodal Generation and Editing: A Survey

#### Kombination mehrerer Suchanfragen

In [134]:
db.delete_collection()

queries = ['multimodal', 'evaluate multimodal', 'benchmark LLM']
max_results = 50

all_docs = []

for q in queries: 
    search = arxiv.Search(query=q, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance)
    docs = create_docs(search)
    all_docs += docs

print(len(all_docs))

chunks = create_chunks(all_docs)
embedding_model = HuggingFaceEmbeddings(model_name= "sentence-transformers/all-MiniLM-L6-v2")
db = create_vector_store(embedding_model, db_name='../../Data/arxiv_db_larger')

/var/folders/j9/lnrs75px2vvdhgh2j143swp40000gn/T/ipykernel_6247/3041715250.py:3: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


50 Artikel geladen.
50 Artikel geladen.
50 Artikel geladen.
150
1410 Chunks erzeugt.
Vektordatenbank erstellt.


- Wir sehen nun Duplikate in der Trefferliste - woran könnte das liegen und an welcher Stelle könnten wir dieses Problem beheben? 
- Hinweis: Es gibt mehrere Ansatzpunkte.

In [135]:
query = 'evaluate multimodal LLMs' 
results = db.similarity_search_with_score(query, k=5)
print_results(results)


 Treffer: Wiki-LLaVA: Hierarchical Retrieval-Augmented Generation for Multimodal LLMs
Multimodal LLMs are the natural evolution of LLMs, and enlarge their capabilities so as to work beyond the pure textual modality. As research is being
Quelle: http://arxiv.org/abs/2404.15406v2
Cosine similarity with query: 0.5436115264892578

 Treffer: Wiki-LLaVA: Hierarchical Retrieval-Augmented Generation for Multimodal LLMs
Multimodal LLMs are the natural evolution of LLMs, and enlarge their capabilities so as to work beyond the pure textual modality. As research is being
Quelle: http://arxiv.org/abs/2404.15406v2
Cosine similarity with query: 0.5436115264892578

 Treffer: LLMs Meet Multimodal Generation and Editing: A Survey
of LLMs in multimodal generation and exhaustively investigate the critical technical components behind these methods and the multimodal datasets
Quelle: http://arxiv.org/abs/2405.19334v2
Cosine similarity with query: 0.5383468866348267

 Treffer: LLMs Meet Multimodal Generatio

### 5.2 Anderes Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# andere Chunking-Methode implementieren
def create_chunks_recursive_split(docs):
    documents = [
        Document(page_content=d["summary"], metadata={"title": d["title"], "url": d["url"]})
        # Wir nutzen hier die Variable docs, die oben von print_hits() zurückgegeben wurde
        for d in docs
    ]

    # default chunk_size = 1000
    # default chunk_overlap = 200
    splitter = RecursiveCharacterTextSplitter()
    chunks = splitter.split_documents(documents)
    print(f"{len(chunks)} Chunks erzeugt.")
    return chunks

In [156]:
# db.delete_collection() # - use this to flush DB

queries = ['multimodal']
max_results = 100

all_docs = []

for q in queries: 
    search = arxiv.Search(query=q, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance)
    docs = create_docs(search)
    all_docs += docs

print(len(all_docs))

chunks = create_chunks_recursive_split(all_docs)
embedding_model = HuggingFaceEmbeddings(model_name= "sentence-transformers/all-MiniLM-L6-v2")
db = create_vector_store(embedding_model, db_name='../../Data/arxiv_db_diff_chunking')

/var/folders/j9/lnrs75px2vvdhgh2j143swp40000gn/T/ipykernel_6247/3041715250.py:3: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


100 Artikel geladen.
100
100 Chunks erzeugt.
Vektordatenbank erstellt.


In [154]:
query = 'evaluate multimodal LLMs' 
results = db.similarity_search_with_score(query, k=5)
print_results(results)


 Treffer: MME-Survey: A Comprehensive Survey on Evaluation of Multimodal LLMs
LLMs, this family of models further develops multimodal perception and reasoning capabilities that are impressive, such as writing code given a flow chart or creating stories based on an image. In the development process, evaluation is critical since it provides intuitive feedback and guidance on improving models. Distinct from the traditional train-eval-test paradigm that only favors a single
Quelle: http://arxiv.org/abs/2411.15296v2
Cosine similarity with query: 0.32605278491973877

 Treffer: LLMs Meet Multimodal Generation and Editing: A Survey
advancements with milestone works in these fields and categorize these studies into LLM-based and CLIP/T5-based methods. Then, we summarize the various roles of LLMs in multimodal generation and exhaustively investigate the critical technical components behind these methods and the multimodal datasets utilized in these studies. Additionally, we dig into tool-augmen

#### 5.3 Filtern nach Jahr

In [164]:
import pandas as pd

def create_docs(search, year_min):
    docs = []
    for result in search.results():
        # result.published hat den Typ datetime, wir können auf .year zugreifen und dann mit dem übergebenen int-Wert vergleichen
        if result.published.year >= year_min:
            docs.append({
                "title": result.title,
                "summary": result.summary,
                "url": result.entry_id,
                "published": result.published
            })

    print(f"{len(docs)} Artikel geladen.")
    return docs

query = 'multimodal'
max_results = 10
search = arxiv.Search(query=query, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance)
# Filtern auf Publikationen aus dem Jahr 2025 (oder später)
docs = create_docs(search, year_min=2025)

/var/folders/j9/lnrs75px2vvdhgh2j143swp40000gn/T/ipykernel_6247/3858534927.py:5: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


3 Artikel geladen.


- An welcher Stelle in der Pipeline könnten wir noch nach dem Publikationsjahr filtern? 
- Was spricht dafür, es bereits bei der Data Collection zu tun? Was spricht vielleicht aber auch dagegen? 